In [1]:
import os
import glob
from google.cloud import bigquery
from google.oauth2 import service_account

In [10]:
CREDENTIALS_FILE = "project-thuhuyen-2023-bigquery.json"   # đường dẫn tới file key JSON
PROJECT_ID       = "project-thuhuyen-2023"    # Google Cloud Project ID
DATASET_ID       = "K312"         # tên dataset trong BigQuery
CSV_FOLDER       = "data"  

In [11]:
def get_client() -> bigquery.Client:
    """Tạo BigQuery client từ service account credentials."""
    credentials = service_account.Credentials.from_service_account_file(
        CREDENTIALS_FILE,
        scopes=["https://www.googleapis.com/auth/cloud-platform"],
    )
    client = bigquery.Client(project=PROJECT_ID, credentials=credentials)
    print(f"✅ Kết nối BigQuery thành công — project: {PROJECT_ID}")
    return client

In [12]:
def ensure_dataset(client: bigquery.Client, dataset_id: str) -> None:
    """Tạo dataset nếu chưa tồn tại."""
    dataset_ref = bigquery.Dataset(f"{PROJECT_ID}.{dataset_id}")
    dataset_ref.location = "US"

    try:
        client.get_dataset(dataset_ref)
        print(f"📂 Dataset '{dataset_id}' đã tồn tại.")
    except Exception:
        client.create_dataset(dataset_ref, exists_ok=True)
        print(f"📂 Đã tạo dataset '{dataset_id}'.")

In [13]:
def upload_csv(client: bigquery.Client, csv_path: str, dataset_id: str) -> None:

    file_name   = os.path.basename(csv_path)          # orders.csv
    table_name  = os.path.splitext(file_name)[0]      # orders
    table_id    = f"{PROJECT_ID}.{dataset_id}.{table_name}"

    job_config = bigquery.LoadJobConfig(
        source_format=bigquery.SourceFormat.CSV,
        skip_leading_rows=1,          # bỏ dòng header
        autodetect=True,              # tự detect schema
        write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
        # Đổi thành WRITE_APPEND để append thay vì overwrite
    )

    print(f"\n⬆️  Đang upload: {file_name} → {table_id}")
    with open(csv_path, "rb") as f:
        job = client.load_table_from_file(f, table_id, job_config=job_config)

    job.result()  # chờ job hoàn thành

    table = client.get_table(table_id)
    print(f"   ✅ Thành công — {table.num_rows:,} dòng, {len(table.schema)} cột")

In [14]:
def upload_all_csv(folder: str, dataset_id: str) -> None:
    """Upload toàn bộ file CSV trong một thư mục."""
    csv_files = glob.glob(os.path.join(folder, "*.csv"))

    if not csv_files:
        print(f"⚠️  Không tìm thấy file CSV trong '{folder}'")
        return

    print(f"\n📋 Tìm thấy {len(csv_files)} file CSV:\n   " + "\n   ".join(csv_files))

    client = get_client()
    ensure_dataset(client, dataset_id)

    success, failed = 0, []
    for csv_path in csv_files:
        try:
            upload_csv(client, csv_path, dataset_id)
            success += 1
        except Exception as e:
            print(f"   ❌ Lỗi với {csv_path}: {e}")
            failed.append(csv_path)

    print(f"\n{'='*50}")
    print(f"✅ Thành công: {success}/{len(csv_files)} file")
    if failed:
        print(f"❌ Thất bại:   {len(failed)} file")
        for f in failed:
            print(f"   - {f}")

In [15]:
def upload_single_csv(csv_path: str, dataset_id: str) -> None:

    client = get_client()
    ensure_dataset(client, dataset_id)
    upload_csv(client, csv_path, dataset_id)

In [17]:
if __name__ == "__main__":
    # --- Tùy chọn 1: Upload toàn bộ CSV trong thư mục ---
    upload_all_csv(CSV_FOLDER, DATASET_ID)


📋 Tìm thấy 5 file CSV:
   data\address.csv
   data\customer.csv
   data\film.csv
   data\film_actor.csv
   data\rental.csv
✅ Kết nối BigQuery thành công — project: project-thuhuyen-2023
📂 Dataset 'K312' đã tồn tại.

⬆️  Đang upload: address.csv → project-thuhuyen-2023.K312.address
   ✅ Thành công — 603 dòng, 6 cột

⬆️  Đang upload: customer.csv → project-thuhuyen-2023.K312.customer
   ✅ Thành công — 599 dòng, 7 cột

⬆️  Đang upload: film.csv → project-thuhuyen-2023.K312.film
   ✅ Thành công — 1,000 dòng, 12 cột

⬆️  Đang upload: film_actor.csv → project-thuhuyen-2023.K312.film_actor
   ✅ Thành công — 5,462 dòng, 5 cột

⬆️  Đang upload: rental.csv → project-thuhuyen-2023.K312.rental
   ✅ Thành công — 16,044 dòng, 9 cột

✅ Thành công: 5/5 file
